# 🤖 Financial Fraud Detection – Model Training & Evaluation

This notebook walks through:
1. Feature engineering & preprocessing
2. Training Logistic Regression, XGBoost, and LightGBM
3. Evaluation with AUC-ROC, Precision-Recall, F1
4. SHAP explainability for the best model

---

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Ready ✓')

## 1. Preprocessing

In [ ]:
from data_preprocessing import preprocess

data = preprocess('../data/improved_fraud_dataset.csv')

print('\nFeature list:')
print(data['feature_names'])

## 2. Train Models

In [ ]:
from model_training import train_models

results = train_models(
    data['X_train'], data['y_train'],
    data['X_test'],  data['y_test'],
)

## 3. Evaluation Plots

In [ ]:
from model_training import (
    plot_roc_curves, plot_pr_curves,
    plot_confusion_matrices, plot_metrics_comparison,
)

plot_roc_curves(results, data['y_test'])
plot_pr_curves(results, data['y_test'])
plot_confusion_matrices(results, data['y_test'])
plot_metrics_comparison(results)

## 4. Metrics Summary Table

In [ ]:
rows = []
for name, res in results.items():
    m = res['metrics']
    rows.append({
        'Model': name,
        'CV AUC (mean)': round(m['cv_auc_mean'], 4),
        'Test AUC-ROC': round(m['test_auc_roc'], 4),
        'Avg Precision': round(m['test_avg_precision'], 4),
        'F1-Score': round(m['test_f1'], 4),
    })

pd.DataFrame(rows)

## 5. SHAP Explainability (LightGBM)

In [ ]:
from explainability import run_explainability

shap_values = run_explainability(
    model=results['LightGBM']['model'],
    X_test=data['X_test'],
    model_name='lightgbm',
    n_samples=3000,
)

## 6. Save Best Model

In [ ]:
from inference import FraudDetector

detector = FraudDetector(
    model=results['LightGBM']['model'],
    scaler=data['scaler'],
    encoders=data['encoders'],
    feature_names=data['feature_names'],
)
detector.save('fraud_detector', directory='../models')
print('Detector saved ✓')

## 7. Demo Inference

In [ ]:
sample = pd.read_csv('../data/improved_fraud_dataset.csv').sample(10, random_state=99)
predictions = detector.predict(sample)
print(predictions.to_string(index=False))